# 02 · FP32 student training  (T4 · main ~1.5–2 h + ablations ~1.5–2 h)

1. **Main run** — MobileFaceNet-FPGA width 0.75 (`configs/student_m.yaml`): ArcFace + cosine distillation
   from the cached iResNet-50 teacher, 15 epochs on all of CASIA-WebFace.
2. **Ablations** — 2k-identity subset, 8 epochs each (`configs/ablations/`): distillation on/off, CosFace / AdaFace,
   64 px input, width 0.5, VGG baseline. Compare them to `abl_ref`, not to the main run.

**Settings:** Accelerator *GPU T4 ×1* (×2 also works — torchrun is used automatically), Internet *On*.
**Inputs:** the CASIA dataset, **output of notebook 00** (prepared data), **output of notebook 01** (teacher embeddings).

Run it with **Save Version → Save & Run All** so it trains in the background (up to 12 h) without the browser open.
If a run is cut off, attach this notebook's previous output as an extra input and run again: it resumes.

In [ ]:
# Setup: clone the repo (code lives in src/fpgaface, notebooks are thin wrappers)
import os
REPO = "https://github.com/sidwastaken07/face-detection.git"
BRANCH = "claude/keen-meitner-ab0d8k"
CODE = "/tmp/face-detection"
if not os.path.exists(CODE):
    !git clone -q -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull -q
%cd {CODE}
!pip install -q -e . onnx onnx2torch
!pip install -q --no-deps facenet-pytorch
# A running kernel does not see a fresh editable install; make the package importable directly.
import sys
if f"{CODE}/src" not in sys.path:
    sys.path.insert(0, f"{CODE}/src")


In [ ]:
import glob, os, torch
DATA = os.path.dirname(glob.glob("/kaggle/input/**/data_manifest.json", recursive=True)[0])
TEACHER = os.path.dirname(glob.glob("/kaggle/input/**/teacher_arcface_r50.f16.npy", recursive=True)[0])
RUNS = "/kaggle/working/runs"
NGPU = torch.cuda.device_count()
LAUNCH = f"torchrun --nproc_per_node {NGPU}" if NGPU > 1 else "python"
print("data:", DATA, "| teacher:", TEACHER, "| GPUs:", NGPU)

def resume_arg(run_name):
    """Resume from a previous version of this notebook attached as input, if any."""
    prev = glob.glob(f"/kaggle/input/**/runs/{run_name}/last.pt", recursive=True)
    return f"--resume {prev[0]}" if prev and not os.path.exists(f"{RUNS}/{run_name}/last.pt") else ""

def train(config, run_name, extra=""):
    !{LAUNCH} scripts/train_student.py --config {config} --data {DATA} --teacher-dir {TEACHER} --out {RUNS} {resume_arg(run_name)} {extra}

In [ ]:
# On-chip memory estimate that motivated the model sizes (first-order; FINN reports replace it in Phase 6)
!python scripts/budget.py --device z7020

## Main run

In [ ]:
train("configs/student_m.yaml", "mfn_m_arcface_kd")

## Ablations (2k identities, 8 epochs each)

In [ ]:
ABLATIONS = ["abl_base", "abl_kd_off", "abl_cosface", "abl_adaface", "abl_res64", "abl_width_s", "abl_vgg"]
# optional extras if time allows: "abl_kd_relational", "abl_width_l"
for name in ABLATIONS:
    run_name = "abl_ref" if name == "abl_base" else name
    train(f"configs/ablations/{name}.yaml", run_name)

## Results

In [ ]:
from IPython.display import Markdown
!python scripts/summarize_runs.py {RUNS} --out /kaggle/working/results_fp32.md
display(Markdown(open("/kaggle/working/results_fp32.md").read()))

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for path in sorted(glob.glob(f"{RUNS}/*/metrics.jsonl")):
    name = path.split("/")[-2]
    df = pd.DataFrame([json.loads(l) for l in open(path)])
    ep = df[df.val_mean.notna()] if "val_mean" in df else df.iloc[:0]
    ax[0].plot(ep.epoch + 1, ep.val_mean * 100, marker="o", label=name)
    st = df[df.loss.notna()] if "loss" in df else df.iloc[:0]
    ax[1].plot(st.elapsed_h, st.ce, label=name, alpha=.7)
ax[0].set(xlabel="epoch", ylabel="mean verification acc (%)", title="LFW / CFP-FP / AgeDB mean")
ax[1].set(xlabel="hours", ylabel="margin CE loss", title="training loss")
ax[0].legend(fontsize=7); plt.show()